In [150]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
import numpy as np
import h5py
import re

# root = "/home/lwu/work/NarrativeGenomics/region_narration/code/pipeline/ARCHS4_LW/refining/data_slash_results/"
# filename = root + 'archs4_gene_human_v2.1_normalized_LW20230126.h5'

filename = "/home/lwu/ARCHS4/Tests/top_10_genesets_per_category/test_cases/test_h5df_data.hdf5"

with h5py.File(filename, 'r') as f:
    logOE_matrix = f['data/qn_matrix'][:] # EDIT NEEDED
    archs_gene_names = f['meta/genes/all_gene_names'][:]
    category_names = f['meta/biosamples/category'][:]

category_names = [x.decode('UTF-8') for x in category_names]
archs_gene_names = [x.decode('UTF-8') for x in archs_gene_names]

# Variables 
category_list  = {
    "Heart", "Atrium", "Cardiac Muscle Fiber", "Pericardium", "Valve", "Ventricle","HUVEC", 
    "Adipose", "Adipocyte", "Myofibroblast", "Osteoblast", "Stromal Cell", 
    "MG63", "K562", "Bone Marrow", "Chondrocyte", "HT1080", "NHDF",
    "HCT116", "HT29", "RKO", "Colon", "Colonic Muscosa", "Intestinal Epithelial Cell", 
    "Small Intestine", "Liver", "Hepatic Stellate Cell", "Hepatocyte", "Primary Liver Cell",
    "HEPG2", "Alpha Cell", "Beta Cell", "Pancreatic Islet", 
    "GSE81547", "PANC1", "Gastric Epithelial Cell", "Esophagus","Granulocyte", "Neutrophil", "Blymphocyte", 
    "Plasma Cell", "Plasmacytoid Dendritic Cell", 
    "Tlymphocyte", "GSM2679484", "JURKAT", "MT4", "NALM6", "REH", "RPMI8226", "Alveolar", 
    "Dendritic", "Kupffer", "Macrophage", "Macroglia",  "THP1", "U937", "JEG3", "Thymus", 
    "Thymocyte", "Spleen",
    "Skin", "Basal Cell", "Fibroblast", "Hair Follicle", "Keratinocyte", "Melanocyte",
    "A431", "HACAT", "HNSCC","Skeletal Muscle", "Myoblast", "Airway Smooth Muscle", "Respiratory Smooth Muscle", 
    "Vascular Smooth Muscle", "LHCNM2","Brain", "DAOY", "IMR32", "SKNSH", "Astrocyte", "Cerebellum", "Cerebral Cortex", 
    "Hypothalamus", "Medula", "Midbrain", "Neuron", "Oligodendrocyte", "Pons", "Spinal Cord", 
    "Thalamus", "Retina", "Motor Neuron", "Sensory Neuron","Lung", "Lung Epithelial", "A549", "H1299", "IMR90", "MRC5", "NCIH460", "NHBE", 
    "Trachea","Breast", "Mammary Gland", "MCF7", "MCF10", "MDAMB231", "MDAMB468", "SKBR3", "T47D", 
    "HELA", "Kidney", "Podocyte", "Renal Cortex", "293F", "FLPIN TREX 293", "HEK293", "C42", 
    "DU145", "LNCAP", "PC3", "Ovary", "Granulosa", "Oocyte", "ES2", "SKOV3", "Testis"
}
category_dict  = {
    "Cardiovascular System" : ["Heart", "Atrium", "Cardiac Muscle Fiber", 
                               "Pericardium", "Valve", "Ventricle",
                               "HUVEC"], 
    
    "Connective Tissue" : ["Adipose", "Adipocyte", "Myofibroblast", "Osteoblast", "Stromal Cell", 
                           "MG63", "K562", "Bone Marrow", "Chondrocyte", "HT1080", "NHDF"],

    "Digestive System": ["HCT116", "HT29", "RKO", "Colon", "Colonic Muscosa", "Intestinal Epithelial Cell", 
                         "Small Intestine", "Liver", "Hepatic Stellate Cell", "Hepatocyte", "Primary Liver Cell",
                         "HEPG2", "Alpha Cell", "Beta Cell", "Pancreatic Islet", 
                         "GSE81547", "PANC1", "Gastric Epithelial Cell", "Esophagus"],
    
    "Immune System" : ["Granulocyte", "Neutrophil", "Blymphocyte", "Plasma Cell", "Plasmacytoid Dendritic Cell", 
                       "Tlymphocyte", "GSM2679484", "JURKAT", "MT4", "NALM6", "REH", "RPMI8226", "Alveolar", 
                       "Dendritic", "Kupffer", "Macrophage", "Macroglia",  "THP1", "U937", "JEG3", "Thymus", 
                       "Thymocyte", "Spleen"],
    
    "Integumentary System" : ["Skin", "Basal Cell", "Fibroblast", "Hair Follicle", "Keratinocyte", "Melanocyte",
                              "A431", "HACAT", "HNSCC"], 
    
    "Muscular System" : ["Skeletal Muscle", "Myoblast", "Airway Smooth Muscle", "Respiratory Smooth Muscle", 
                         "Vascular Smooth Muscle", "LHCNM2"], 
    
    "Nervous System" : ["Brain", "DAOY", "IMR32", "SKNSH", "Astrocyte", "Cerebellum", "Cerebral Cortex", 
                        "Hypothalamus", "Medula", "Midbrain", "Neuron", "Oligodendrocyte", "Pons", "Spinal Cord", 
                        "Thalamus", "Retina", "Motor Neuron", "Sensory Neuron"], 
    
    "Respiratory System" : ["Lung", "Lung Epithelial", "A549", "H1299", "IMR90", "MRC5", "NCIH460", "NHBE", 
                            "Trachea"], 
    
    "Urogenital System" : ["Breast", "Mammary Gland", "MCF7", "MCF10", "MDAMB231", "MDAMB468", "SKBR3", "T47D", 
                           "HELA", "Kidney", "Podocyte", "Renal Cortex", "293F", "FLPIN TREX 293", "HEK293", "C42", 
                           "DU145", "LNCAP", "PC3", "Ovary", "Granulosa", "Oocyte", "ES2", "SKOV3", "Testis"]
}


# folder = "/home/lwu/ARCHS4/Tests/Check If Biosample Categorys Align With Genesets/"
# filename = folder + "h.c1.c2.all.v7.5.1.symbols.gmt"

filename = "/home/lwu/ARCHS4/Tests/top_10_genesets_per_category/test_cases/test_geneset.gmt"

In [151]:
# Get category-sample indexes
all_cat_idxs = {}
for system in category_dict:
    print(system)
    temp_dict = {}
    for element in category_dict[system]:
        indexes = []
        for i, name in enumerate(category_names):
            if element.lower() in name.lower():
                indexes.append(i)
        temp_dict[element] = indexes
    all_cat_idxs[system] = temp_dict

Cardiovascular System
Connective Tissue
Digestive System
Immune System
Integumentary System
Muscular System
Nervous System
Respiratory System
Urogenital System


In [92]:
all_cat_idxs

{'Cardiovascular System': {'Heart': [0, 1, 4],
  'Atrium': [2],
  'Cardiac Muscle Fiber': [],
  'Pericardium': [],
  'Valve': [],
  'Ventricle': [],
  'HUVEC': []},
 'Connective Tissue': {'Adipose': [],
  'Adipocyte': [],
  'Myofibroblast': [],
  'Osteoblast': [],
  'Stromal Cell': [],
  'MG63': [],
  'K562': [],
  'Bone Marrow': [],
  'Chondrocyte': [],
  'HT1080': [],
  'NHDF': []},
 'Digestive System': {'HCT116': [],
  'HT29': [],
  'RKO': [],
  'Colon': [],
  'Colonic Muscosa': [],
  'Intestinal Epithelial Cell': [],
  'Small Intestine': [],
  'Liver': [],
  'Hepatic Stellate Cell': [],
  'Hepatocyte': [],
  'Primary Liver Cell': [],
  'HEPG2': [],
  'Alpha Cell': [5],
  'Beta Cell': [],
  'Pancreatic Islet': [],
  'GSE81547': [],
  'PANC1': [],
  'Gastric Epithelial Cell': [],
  'Esophagus': []},
 'Immune System': {'Granulocyte': [],
  'Neutrophil': [],
  'Blymphocyte': [],
  'Plasma Cell': [],
  'Plasmacytoid Dendritic Cell': [],
  'Tlymphocyte': [],
  'GSM2679484': [],
  'JURKAT

In [72]:
len(all_cat_idxs)

127

---
# Functions

In [45]:
def get_archs4_gene_data(gene_names):
    gene_indexes = []
    new_gene_names = []
    for gene in gene_names:
        try:
            index = archs_gene_names.index(gene)
            gene_indexes.append(index)
            new_gene_names.append(gene)
        except ValueError:
            pass
    return new_gene_names, gene_indexes

gene_names, gene_indexes = get_archs4_gene_data(gene_names)

In [154]:
category_dict  = {
    "Cardiovascular System" : ["heart", "atrium"], 
    
    "Connective Tissue" : ["adipose", "adipocyte"]}

system_biosampleMatrix = {}
system_biosampleMean_matrix = {}

for system, tier1_list in all_cat_idxs.items():
    temp_dict = {}
    temp_dict2 = {}
    for tier1_cat in tier1_list:
        idx_list = tier1_list[tier1_cat]
        
        matrix = extracted_matrix[:,idx_list ].tolist()
        print(matrix)
        avg = np.mean(matrix, axis=0)
        #print("Avg:", avg)
        temp_dict[tier1_cat] = avg
        
        tier1_avg = np.mean(avg, axis=0)
        #print("Avge2", tier1_avg)
        temp_dict2[tier1_cat] = tier1_avg
        #print("Temp dict2", temp_dict2)
        
    system_biosampleMatrix[system] = temp_dict
    system_biosampleMean_matrix[system] = temp_dict2
print(system_biosampleMean_matrix)
#         print(gene_row)
        
        
            
#             print(system)
#             #print(tier1_cat)
# #     for tier_cat in tier1_list:
# #         for idx, biosample in enumerate(category_names):
# #             if tier_cat.replace(" ", "").lower() in biosample.replace(" ", "").lower():
# #                 category_cols.append(idx)

# #         category_biosampleMatrix[category] = extracted_matrix[:, category_cols].tolist()
        
# category_biosampleMatrix

[[1, 2, 5], [11, 12, 15], [21, 22, 25], [41, 42, 45]]
[[3], [13], [23], [43]]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[6], [16], [26], [46]]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], []]
[[], [], [], [

In [171]:
category_dict  = {
    "Cardiovascular System" : ["heart", "atrium"], 
    
    "Connective Tissue" : ["adipose", "adipocyte"]}

system_biosampleMatrix = {}
system_biosampleMean_matrix = {}

for system, tier1_list in all_cat_idxs.items():
    temp_dict = {}
    temp_dict2 = {}
    for tier1_cat in tier1_list:
        idx_list = tier1_list[tier1_cat]
        
        matrix = extracted_matrix[:,idx_list ].tolist()
        avg = np.mean(matrix, axis=0)
        #print("Avg:", avg)
        temp_dict[tier1_cat] = avg
        
        tier1_avg = np.mean(avg, axis=0)
        #print("Avge2", tier1_avg)
        temp_dict2[tier1_cat] = tier1_avg
        #print("Temp dict2", temp_dict2)
        
    system_biosampleMatrix[system] = temp_dict
    system_biosampleMean_matrix[system] = temp_dict2
print(system_biosampleMatrix)
#         print(gene_row)
        
        
            
#             print(system)
#             #print(tier1_cat)
# #     for tier_cat in tier1_list:
# #         for idx, biosample in enumerate(category_names):
# #             if tier_cat.replace(" ", "").lower() in biosample.replace(" ", "").lower():
# #                 category_cols.append(idx)

# #         category_biosampleMatrix[category] = extracted_matrix[:, category_cols].tolist()
        
# category_biosampleMatrix

{'Cardiovascular System': {'Heart': array([18.5, 19.5, 22.5]), 'Atrium': array([20.5]), 'Cardiac Muscle Fiber': array([], dtype=float64), 'Pericardium': array([], dtype=float64), 'Valve': array([], dtype=float64), 'Ventricle': array([], dtype=float64), 'HUVEC': array([], dtype=float64)}, 'Connective Tissue': {'Adipose': array([], dtype=float64), 'Adipocyte': array([], dtype=float64), 'Myofibroblast': array([], dtype=float64), 'Osteoblast': array([], dtype=float64), 'Stromal Cell': array([], dtype=float64), 'MG63': array([], dtype=float64), 'K562': array([], dtype=float64), 'Bone Marrow': array([], dtype=float64), 'Chondrocyte': array([], dtype=float64), 'HT1080': array([], dtype=float64), 'NHDF': array([], dtype=float64)}, 'Digestive System': {'HCT116': array([], dtype=float64), 'HT29': array([], dtype=float64), 'RKO': array([], dtype=float64), 'Colon': array([], dtype=float64), 'Colonic Muscosa': array([], dtype=float64), 'Intestinal Epithelial Cell': array([], dtype=float64), 'Small 

In [180]:
system_biosampleMatrix = {}

for system, tier1_list in all_cat_idxs.items():
    for tier1_cat in tier1_list:
        idx_list = tier1_list[tier1_cat]

        matrix = extracted_matrix[:,idx_list ].tolist()
        avg = np.mean(matrix, axis=0)
        temp_dict[tier1_cat] = avg

        tier1_avg = np.mean(avg, axis=0)
        system_biosampleMatrix[tier1_cat] = tier1_avg
    system_biosampleMatrix 
print("Temp dict2", system_biosampleMatrix)



Temp dict2 {'Heart': 20.166666666666668, 'Atrium': 20.5, 'Cardiac Muscle Fiber': nan, 'Pericardium': nan, 'Valve': nan, 'Ventricle': nan, 'HUVEC': nan, 'Adipose': nan, 'Adipocyte': nan, 'Myofibroblast': nan, 'Osteoblast': nan, 'Stromal Cell': nan, 'MG63': nan, 'K562': nan, 'Bone Marrow': nan, 'Chondrocyte': nan, 'HT1080': nan, 'NHDF': nan, 'HCT116': nan, 'HT29': nan, 'RKO': nan, 'Colon': nan, 'Colonic Muscosa': nan, 'Intestinal Epithelial Cell': nan, 'Small Intestine': nan, 'Liver': nan, 'Hepatic Stellate Cell': nan, 'Hepatocyte': nan, 'Primary Liver Cell': nan, 'HEPG2': nan, 'Alpha Cell': 23.5, 'Beta Cell': nan, 'Pancreatic Islet': nan, 'GSE81547': nan, 'PANC1': nan, 'Gastric Epithelial Cell': nan, 'Esophagus': nan, 'Granulocyte': nan, 'Neutrophil': nan, 'Blymphocyte': nan, 'Plasma Cell': nan, 'Plasmacytoid Dendritic Cell': nan, 'Tlymphocyte': nan, 'GSM2679484': nan, 'JURKAT': nan, 'MT4': nan, 'NALM6': nan, 'REH': nan, 'RPMI8226': nan, 'Alveolar': nan, 'Dendritic': nan, 'Kupffer': nan

In [181]:
def get_category_biosampleMatrix(extracted_matrix):
    
    system_biosampleMatrix = {}

    for system, tier1_list in all_cat_idxs.items():
        for tier1_cat in tier1_list:
            idx_list = tier1_list[tier1_cat]

            matrix = extracted_matrix[:,idx_list ].tolist()
            avg = np.mean(matrix, axis=0)
            temp_dict[tier1_cat] = avg

            tier1_avg = np.mean(avg, axis=0)
            system_biosampleMatrix[tier1_cat] = tier1_avg
    return system_biosampleMatrix

In [61]:
def get_geneset_category_matrix(dictionary):
    
    value_list = []
    for subdict in dictionary.values():
        for values in subdict.values():
            value_list.append(values)

    matrix = np.vstack(value_list)
    return matrix


In [73]:
def get_geneset_biosample_means(extracted_matrix): 
    tier1_category_means = {}

    for category, subcategories in category_dict.items():
        subcategory_dict = {}
        for subcategory in subcategories:
            subcategory_indices = [i for i, sample in enumerate(category_names) if 
                                   subcategory.replace(" ", "").lower() in sample.replace(" ", "").lower()]
            if subcategory_indices:
                submatrix = extracted_matrix[:,subcategory_indices]
                subcategory_avg = np.mean(submatrix, axis=1)
                subcategory_dict[subcategory] = subcategory_avg.tolist()
        if subcategory_dict:
            tier1_category_means[category] = subcategory_dict
            
        #print(subcategory_indices)
            
    return tier1_category_means


In [48]:
logOE_matrix

array([[ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10],
       [11, 12, 13, 14, 15, 16, 17, 18, 19, 20],
       [21, 22, 23, 24, 25, 26, 27, 28, 29, 30],
       [31, 32, 33, 34, 35, 36, 37, 38, 39, 40],
       [41, 42, 43, 44, 45, 46, 47, 48, 49, 50]])

In [183]:
set_geneset_name = "HALLMARK_TNFA_SIGNALING_VIA_NFKB"

all_genesets_matrix_mean = []
all_genesets_matrix_oe = []
geneset_names = []
gene_names = []


with open(filename, 'r', ) as file:
    
    for line in file:
        #print(line)
        line = line.strip().split('\t')
        geneset_names.append(line[0])
        #print(line[0])
        
        if line[0] == set_geneset_name: 
            
            gene_names = line[2:]
            #print(geneset_names, ":", gene_names)

            gene_names, gene_indexes = get_archs4_gene_data(gene_names)
#             print(gene_names, "\n", gene_indexes)

            extracted_matrix = logOE_matrix[gene_indexes, :] 
            #print(extracted_matrix)
            #print("Extracted Matrix Shape:", extracted_matrix.shape)

            category_biosampleMatrix = get_category_biosampleMatrix(extracted_matrix)
            print("----- Result ------")
            print(category_biosampleMatrix)
            
# #             tier1_category_means = get_geneset_biosample_means(extracted_matrix)
# #             print("----- Result2 ------")
# #             print(tier1_category_means)
            
#             geneset_means_matrix = get_geneset_category_matrix(tier1_category_means)
#             geneset_means_matrix = geneset_means_matrix.T
            
#             print("----- Result3 ------")
#             print(geneset_means_matrix)

----- Result ------
{'Heart': 20.166666666666668, 'Atrium': 20.5, 'Cardiac Muscle Fiber': nan, 'Pericardium': nan, 'Valve': nan, 'Ventricle': nan, 'HUVEC': nan, 'Adipose': nan, 'Adipocyte': nan, 'Myofibroblast': nan, 'Osteoblast': nan, 'Stromal Cell': nan, 'MG63': nan, 'K562': nan, 'Bone Marrow': nan, 'Chondrocyte': nan, 'HT1080': nan, 'NHDF': nan, 'HCT116': nan, 'HT29': nan, 'RKO': nan, 'Colon': nan, 'Colonic Muscosa': nan, 'Intestinal Epithelial Cell': nan, 'Small Intestine': nan, 'Liver': nan, 'Hepatic Stellate Cell': nan, 'Hepatocyte': nan, 'Primary Liver Cell': nan, 'HEPG2': nan, 'Alpha Cell': 23.5, 'Beta Cell': nan, 'Pancreatic Islet': nan, 'GSE81547': nan, 'PANC1': nan, 'Gastric Epithelial Cell': nan, 'Esophagus': nan, 'Granulocyte': nan, 'Neutrophil': nan, 'Blymphocyte': nan, 'Plasma Cell': nan, 'Plasmacytoid Dendritic Cell': nan, 'Tlymphocyte': nan, 'GSM2679484': nan, 'JURKAT': nan, 'MT4': nan, 'NALM6': nan, 'REH': nan, 'RPMI8226': nan, 'Alveolar': nan, 'Dendritic': nan, 'Kupf

In [186]:
from IPython.display import display

df = pd.DataFrame(category_biosampleMatrix.values(), index=geneset_names)
df.columns = category_list
display(df)

ValueError: Shape of passed values is (127, 1), indices imply (3, 1)

In [55]:
category_names = ['Heart', 'Heart', 'Atrium', 'Skin', 'Heart', 'Alpha Cell', 'Testis', 'Lung', "lung", "lung3"]

In [184]:
my_dict = {'a': 1, 'b': 2, 'c': 3}

# Get list of keys
keys_list = list(my_dict.keys())
print(keys_list)  # Output: ['a', 'b', 'c']

# Get list of values
values_list = list(my_dict.values())
print(values_list)  # Output: [1, 2, 3]


['a', 'b', 'c']
[1, 2, 3]
